<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/DEV_Funcion_GobiernoNacional_LOR_SMN_UCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN (ejecutar una vez por sesión) ═══
!pip install requests beautifulsoup4 pandas numpy tqdm google-api-python-client --quiet
print("✅ Librerías listas")
import os, re, time, random, zipfile, unicodedata
import requests
import numpy as np
import pandas as pd
from datetime import date
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from IPython.display import display

In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN, EXTRACCIÓN Y CIERRE ═══
# ── 1. CONFIGURACIÓN ─────────────────────────────────────────────────────────
AÑO = 2025
MES_INICIO = 1                 # primer mes del bucle
MES_FIN = None                 # None = último mes disponible
INCLUIR_MES_EN_CURSO = False   # False = solo meses cerrados (regla del scraper anterior)

CONTROL_ANUAL = True           # compara contra la vista anual de Consulta Amigable
SUBIR_A_DRIVE = True           # sube cada CSV a la carpeta de Drive de arriba
DRIVE_FOLDER_ID = "1B7Z4_4Bop8WK33YvFZ8pX-xCQBmaTHnk"
DRIVE_FOLDER_CONTROL_ID = "1v2fa0FKJFi_0l1uGECJjYw8TvHPCDTbo"   # archivos CONTROL (control anual)
DESCARGA_FINAL = "ninguna"     # "zip" | "individual" | "ninguna"

# ── Nivel de detalle ─────────────────────────────────────────────────────────
#   "DEPARTAMENTO" -> Departamento > Función          (1 consulta por región y mes: rápido)
#   "PLIEGO"       -> Departamento > Sector > Pliego > Función
#   "EJECUTORA"    -> Departamento > Sector > Pliego > Unidad Ejecutora > Función
#                     (en el Gobierno Nacional hay muchos sectores y ejecutoras: puede
#                     tomar más de una hora el año completo)
NIVEL_DETALLE = "DEPARTAMENTO"

# ── Cómo arma la URL Consulta Amigable ───────────────────────────────────────
# Cada filtro es una clave numérica (24=Genérica, 23=Mes, 1=Nivel, 5=Departamento,
# 21=Departamento (Meta)…). La PRIMERA clave con valor vacío es la columna que se
# lista en la tabla de detalle; todo lo que viene después se ignora.
# Ej.: …&1=R&21=16&8=  -> lista las funciones de los GR en Loreto.
NIVEL_GOBIERNO = "E"                          # E = Gobierno Nacional (R = Regionales, M = Locales)
NIVEL_ETIQUETA = "E: GOBIERNO NACIONAL"
NIVEL_CORTO = "GobNacional"                   # se usa en los nombres de archivo
GENERICA = "6-2-6"                            # Genérica 6-26
CLAVES_DEPTO = ["21"]                         # Departamento (Meta), como en la web ("5" = Departamento)
CLAVE = {"Sector": "2", "Pliego": "3", "Ejecutora": "4", "Función": "8"}
PARAMS_EXTRA = {}   # si F12>Network muestra otros parámetros, se sobreescriben aquí

PAUSA = (0.4, 1.0)             # segundos aleatorios entre consultas (cortesía con el servidor)

# ── Meses válidos ────────────────────────────────────────────────────────────
hoy = date.today()
if AÑO > hoy.year:
    raise ValueError("Año inválido: no puede ser futuro.")
if AÑO < hoy.year:
    meses_validos = list(range(1, 13))
else:
    tope = hoy.month if INCLUIR_MES_EN_CURSO else hoy.month - 1
    if tope < 1:
        raise ValueError("No hay meses cerrados aún en este año.")
    meses_validos = list(range(1, tope + 1))
ultimo_valido = meses_validos[-1]
if MES_FIN is None:
    MES_FIN = ultimo_valido
if not (1 <= MES_INICIO <= MES_FIN <= ultimo_valido):
    raise ValueError(f"Rango inválido ({MES_INICIO}-{MES_FIN}). Disponibles: 1 a {ultimo_valido}.")
meses = [str(m) for m in range(MES_INICIO, MES_FIN + 1)]

# ── Catálogos ────────────────────────────────────────────────────────────────
departamentos = [
    {"codigo": "16", "departamento": "16: LORETO"},
    {"codigo": "22", "departamento": "22: SAN MARTÍN"},
    {"codigo": "25", "departamento": "25: UCAYALI"},
]
nombres_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto",
                 "Setiembre","Octubre","Noviembre","Diciembre"]
meses_map  = {str(i): f"{i}: {nombres_meses[i-1]}" for i in range(1, 13)}
deptos_map = {d["codigo"]: d["departamento"] for d in departamentos}

RUTAS = {
    "EJECUTORA":    ["Sector", "Pliego", "Ejecutora", "Función"],
    "PLIEGO":       ["Sector", "Pliego", "Función"],
    "DEPARTAMENTO": ["Función"],
}
if NIVEL_DETALLE not in RUTAS:
    raise ValueError(f"NIVEL_DETALLE debe ser uno de {list(RUTAS)}")
NIVELES = RUTAS[NIVEL_DETALLE]
COLUMNA_DE = {"Sector": "SECTOR", "Pliego": "PLIEGO", "Ejecutora": "UNIDAD_EJECUTORA",
              "Función": "FUNCION"}

ruta_guardado = "/content/Inversion_Funcion.GobiernoNacional"
os.makedirs(ruta_guardado, exist_ok=True)
SUFIJO_REGIONES = "LOR - SMN - UCA"

def etiqueta_meses(ms):
    nums = sorted(int(m) for m in ms)
    if len(nums) == 1:
        return f"Mes {nums[0]:02d}"
    if nums == list(range(nums[0], nums[-1] + 1)):
        return f"Meses {nums[0]:02d}-{nums[-1]:02d}"
    return "Meses " + ",".join(f"{m:02d}" for m in nums)

def nombre_archivo_salida(año, ms):
    return f"DEV Funcion.{NIVEL_CORTO} {año} - {etiqueta_meses(ms)} - {SUFIJO_REGIONES}.csv"

# ENTIDAD = "GOBIERNO NACIONAL" solo si no se baja a Pliego (sería la misma columna)
COLUMNAS_SALIDA = (["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO"]
                   + ([] if "Pliego" in NIVELES else ["ENTIDAD"])
                   + [COLUMNA_DE[n] for n in NIVELES if n != "Sector"]
                   + ["DEVENGADO"])
print(f"Año {AÑO} -> meses a procesar: {', '.join(meses)}  |  regiones: "
      f"{', '.join(d['departamento'] for d in departamentos)}")
print(f"Ruta por región: Departamento > {' > '.join(NIVELES)}")

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
drive_subidos, drive_fallidos = [], []
STATUS_REINTENTABLES = {403, 429, 500, 502, 503, 504}

def conectar_drive(folder_ids):
    """Autentica y verifica permiso de Editor en cada carpeta."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    auth.authenticate_user()
    servicio = build("drive", "v3", cache_discovery=False)
    for folder_id in dict.fromkeys(folder_ids):
        try:
            carpeta = servicio.files().get(
                fileId=folder_id, fields="id,name,mimeType,capabilities(canAddChildren)",
                supportsAllDrives=True).execute()
        except HttpError as e:
            raise PermissionError(f"Sin acceso a la carpeta de Drive {folder_id} (HTTP {e.resp.status}).") from e
        if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
            raise ValueError(f"El ID {folder_id} no corresponde a una carpeta de Drive.")
        if not carpeta.get("capabilities", {}).get("canAddChildren", False):
            raise PermissionError(f"Sin permiso de Editor en '{carpeta['name']}'.")
        print(f"☁️  Drive conectado -> carpeta: '{carpeta['name']}'")
    print()
    return servicio

def subir_a_drive(servicio, ruta_local, folder_id, max_reintentos=5):
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload
    nombre = os.path.basename(ruta_local)
    mime = "application/zip" if nombre.lower().endswith(".zip") else "text/csv"
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")
    ultimo_error = ""
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(
                q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                fields="files(id)", supportsAllDrives=True,
                includeItemsFromAllDrives=True).execute().get("files", [])
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:
                servicio.files().update(fileId=existentes[0]["id"], media_body=media,
                                        fields="id", supportsAllDrives=True).execute()
                accion = "actualizado"
            else:
                servicio.files().create(body={"name": nombre, "parents": [folder_id]},
                                        media_body=media, fields="id",
                                        supportsAllDrives=True).execute()
                accion = "creado"
            drive_subidos.append(nombre)
            print(f"   ☁️ Drive ({accion}): {nombre}")
            return True
        except HttpError as e:
            ultimo_error = f"HTTP {e.resp.status}"
            if e.resp.status not in STATUS_REINTENTABLES:
                break
        except Exception as e:
            ultimo_error = str(e)
        if intento < max_reintentos:
            espera = 2 ** intento + random.random()
            print(f"   ↻ Reintento {intento}/{max_reintentos - 1} en {espera:.1f}s ({ultimo_error})")
            time.sleep(espera)
    drive_fallidos.append((nombre, ultimo_error))
    print(f"   ❌ No se pudo subir {nombre} ({ultimo_error}). Sigue en {ruta_local}")
    return False



# ── 3. NAVEGACIÓN EN CONSULTA AMIGABLE (solo por URL) ────────────────────────
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
sesion = requests.Session()
sesion.headers.update({"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                       "Referer": "https://apps5.mineco.gob.pe/transparencia/mensual/default.aspx"})

# Variantes de URL que se prueban una vez (calibrar) hasta que una traiga la
# tabla de detalle. La primera imita al navegador (iframe de default.aspx).
FORMATOS = [
    {"nombre": "navegador GET", "metodo": "GET",
     "pre": {"_tgt": "frame", "_uhc": "yes"}, "post": {"ap": "ActProy"}},
    {"nombre": "GET + ap", "metodo": "GET", "pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST + ap", "metodo": "POST", "pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST original", "metodo": "POST", "pre": {}, "post": {}},
]
FORMATO = None                 # lo fija calibrar()

def a_numero(txt):
    t = (txt or "").replace(",", "").replace("\xa0", "").strip()
    if t in ("", "-"):
        return 0.0
    try:
        return float(t)
    except ValueError:
        return np.nan

def normalizar(txt):
    t = unicodedata.normalize("NFKD", txt or "")
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

_FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|"
                     r"provincia|distrito|funci[oó]n|mes\b|trimestre|sector|pliego|ejecutora|"
                     r"unidad ejecutora|fuente|rubro|categor|producto|divisi[oó]n|"
                     r"grupo funcional|sub-?gen[eé]rica)", re.I)
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

class Pagina:
    """Respuesta de Consulta Amigable ya parseada.
    filtros: breadcrumb [(etiqueta, devengado)]
    filas  : detalle    [(valor_para_url, etiqueta, devengado)]"""
    def __init__(self, html, url):
        self.url, self.html = url, html
        s = BeautifulSoup(html, "html.parser")
        self.tiene_tabla = s.find("table", class_="Data") is not None
        self.filtros, self.filas = [], []
        for tr in s.find_all("tr"):
            if tr.find("table"):
                continue
            celdas = tr.find_all("td")
            if len(celdas) < 8:
                continue
            dev = a_numero(celdas[-3].get_text(strip=True))   # … Devengado | Girado | Avance %
            if dev != dev:                                     # NaN -> no es fila numérica
                continue
            textos = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:3]]
            etiqueta = next((t for t in textos if t), "")
            radio = tr.find("input", attrs={"type": "radio"})
            if _FILTRO.match(etiqueta) and radio is None:
                self.filtros.append((etiqueta, dev))
                continue
            m = _CODIGO.match(etiqueta)
            valor = (radio.get("value") if radio is not None else None) or (m.group(1) if m else None)
            if valor is None:
                continue
            self.filas.append((valor.strip(), etiqueta, dev))

def armar_params(anio, mes, depto, ruta, agrupar, formato):
    """ruta = [(clave, valor), …] de filtros ya elegidos; agrupar = clave a listar."""
    p = dict(formato["pre"])
    p.update({"0": "", "24": GENERICA})
    if mes:
        p["23"] = mes                  # vista anual: se OMITE (vacío cortaría la ruta)
    p["1"] = NIVEL_GOBIERNO
    for k in CLAVES_DEPTO:
        p[k] = depto
    for k, v in ruta:
        p[k] = v
    p[agrupar] = ""
    p["y"] = anio
    p.update(formato["post"])
    p.update(PARAMS_EXTRA)
    return {k: v for k, v in p.items() if v is not None}

def consultar(anio, mes, depto, ruta, agrupar, formato=None, reintentos=4):
    formato = formato or FORMATO
    params = armar_params(anio, mes, depto, ruta, agrupar, formato)
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion.request(formato["metodo"], URL_BASE, params=params, timeout=40)
            if r.status_code == 200:
                p = Pagina(r.content, r.url)
                if p.tiene_tabla:
                    return p, None
                ultimo = "SIN_TABLA"
            else:
                ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** i + random.random()))
    return None, ultimo

def calibrar(anio, mes, depto, verbose=True):
    """Prueba los FORMATOS hasta que uno liste funciones del departamento."""
    global FORMATO
    for f in FORMATOS:
        p, err = consultar(anio, mes, depto, [], CLAVE["Función"], formato=f, reintentos=2)
        n = len(p.filas) if p else 0
        if verbose:
            print(f"   · {f['nombre']:<15} -> {err or f'{n} filas de detalle'}")
        if p and n and not validar_departamento(p, depto):
            FORMATO = f
            return p
        time.sleep(random.uniform(*PAUSA))
    return None

def cuadre(p, contexto):
    """Suma del detalle vs la última fila del breadcrumb (el filtro aplicado)."""
    if not p.filtros or not p.filas:
        return None
    total = p.filtros[-1][1]
    suma = sum(v for _, _, v in p.filas)
    if abs(suma - total) > max(1, len(p.filas)):
        return f"CUADRE {contexto}: suma {suma:,.0f} vs {p.filtros[-1][0][:40]} {total:,.0f}"
    return None

def validar_departamento(p, depto):
    errores = []
    etiquetas = [normalizar(e) for e, _ in p.filtros]
    if not any(e.startswith("nivel de gobierno") and f" {NIVEL_GOBIERNO.lower()}:" in e for e in etiquetas):
        errores.append(f"la página no está filtrada por {NIVEL_ETIQUETA}")
    if not any(e.startswith("departamento") and f"{depto}:" in e for e in etiquetas):
        errores.append(f"la página no corresponde al departamento {depto}")
    return errores

USAR_CODIGO_ETIQUETA = set()   # claves donde el código de la etiqueta funcionó mejor que el radio

def codigos_de(etiqueta):
    """Códigos posibles de una etiqueta: '001-861: REGION LORETO…' -> ['001-861', '861']."""
    m = _CODIGO.match(etiqueta)
    if not m:
        return []
    cod = m.group(1)
    return [cod] + ([cod.split("-")[-1]] if "-" in cod else [])

def abrir_nivel(anio, mes, depto, ruta, nivel, candidatos=(None,), esperado=None):
    """Consulta la lista de `nivel` bajo `ruta`. Si la última fila elegida se
    puede expresar con varios códigos (valor del radio o código de la etiqueta),
    prueba cada uno hasta que el breadcrumb confirme que el filtro se aplicó."""
    ultimo_err = ""
    for cand in candidatos:
        r = list(ruta) if cand is None else list(ruta[:-1]) + [(ruta[-1][0], cand)]
        p, err = consultar(anio, mes, depto, r, CLAVE[nivel])
        time.sleep(random.uniform(*PAUSA))
        if err:
            ultimo_err = err
            continue
        if esperado is None:
            return p, r, None
        fin = p.filtros[-1][0] if p.filtros else ""
        if normalizar(esperado) in normalizar(fin):
            if cand is not None and cand != candidatos[0]:
                USAR_CODIGO_ETIQUETA.add(ruta[-1][0])
            return p, r, None
        ultimo_err = (f"la URL no aplicó el filtro '{esperado[:40]}' "
                      f"(breadcrumb termina en '{fin[:50]}'; código probado: {cand})")
    return None, None, ultimo_err

def recorrer(anio, mes, depto, niveles, ruta=(), nombres=(), candidatos=(None,)):
    """Baja nivel por nivel. Devuelve (hojas, errores, avisos);
    hojas = [(nombres_de_la_ruta, etiqueta, devengado)] del último nivel."""
    nivel = niveles[0]
    contexto = " > ".join(nombres) or "departamento"
    p, ruta, err = abrir_nivel(anio, mes, depto, ruta, nivel, candidatos,
                               nombres[-1] if nombres else None)
    if err:
        return [], [f"{nivel} en {contexto}: {err}"], []
    if not p.filas:
        if p.filtros and p.filtros[-1][1] == 0:
            return [], [], []
        return [], [f"{nivel} en {contexto}: sin filas de detalle"], []
    avisos = [a for a in [cuadre(p, f"{nivel} en {contexto}")] if a]
    if len(niveles) == 1:
        return [(nombres, et, dev) for _, et, dev in p.filas], [], avisos
    hojas, errores = [], []
    for valor, etiqueta, dev in p.filas:
        if dev == 0:                       # nada que bajar
            continue
        cands = [valor] + codigos_de(etiqueta)
        if CLAVE[nivel] in USAR_CODIGO_ETIQUETA:
            cands.reverse()
        cands = tuple(dict.fromkeys(cands))
        h, e, a = recorrer(anio, mes, depto, niveles[1:], tuple(ruta) + ((CLAVE[nivel], valor),),
                           nombres + (etiqueta,), cands)
        hojas += h; errores += e; avisos += a
    return hojas, errores, avisos

def extraer(anio, mes, depto, niveles=NIVELES):
    """-> (registros, error, aviso) para una región y un mes."""
    if FORMATO is None and calibrar(anio, mes or "1", depto, verbose=False) is None:
        return [], "CALIBRACIÓN: ningún formato de URL trae la tabla de detalle", None
    hojas, errores, avisos = recorrer(anio, mes, depto, niveles)
    regs = []
    for nombres, etiqueta, dev in hojas:
        m = _CODIGO.match(etiqueta)
        r = {"AÑO": anio, "MES": meses_map.get(mes, "ANUAL"), "NIVEL_GOBIERNO": NIVEL_ETIQUETA,
             "DEPARTAMENTO": deptos_map[depto], "COD_DEPTO": depto,
             "COD_FUNCION": m.group(1) if m else etiqueta[:2], "DEVENGADO": dev}
        for n, valor in zip(niveles[:-1], nombres):
            r[COLUMNA_DE[n]] = valor
        r[COLUMNA_DE[niveles[-1]]] = etiqueta
        r["ENTIDAD"] = r.get("PLIEGO", "GOBIERNO NACIONAL")
        regs.append(r)
    return (regs, ("; ".join(errores)) if errores else None,
            ("; ".join(avisos) if avisos else None))

# ── 4. CONTROL ANUAL (año x departamento x función x devengado) ──────────────
def control_anual(df_total, anio, mes_ultimo, completo=True):
    """Compara lo raspado contra la vista anual de Consulta Amigable (sin filtro de mes).
    Detecta además si el Devengado de cada mes es MENSUAL (la suma de meses cuadra)
    o ACUMULADO (el último mes cuadra). Si el rango de meses no cubre el año
    (completo=False), la referencia anual incluye meses no raspados: ESTADO = PARCIAL."""
    filas = []
    for d in departamentos:
        regs, err, _ = extraer(anio, "", d["codigo"], niveles=["Función"])
        if err or not regs:
            print(f"   ❌ Referencia anual {d['departamento']}: {err or 'sin filas'}")
            continue
        ref = {r["COD_FUNCION"]: r["DEVENGADO"] for r in regs}
        sub = df_total[df_total["COD_DEPTO"] == d["codigo"]]
        suma = sub.groupby("COD_FUNCION")["DEVENGADO"].sum()
        ult = sub[sub["MES"] == meses_map[mes_ultimo]].groupby("COD_FUNCION")["DEVENGADO"].sum()
        for cod in sorted(set(ref) | set(suma.index)):
            filas.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"], "COD_FUNCION": cod,
                          "DEV_REF_ANUAL": ref.get(cod, 0.0),
                          "DEV_SUMA_MESES": float(suma.get(cod, 0.0)),
                          "DEV_ULTIMO_MES": float(ult.get(cod, 0.0))})
    ctl = pd.DataFrame(filas)
    if ctl.empty:
        return ctl, "SIN REFERENCIA"
    tol = max(5, len(meses) * 5)
    ctl["DIF_SUMA"] = ctl["DEV_SUMA_MESES"] - ctl["DEV_REF_ANUAL"]
    ctl["DIF_ULTIMO"] = ctl["DEV_ULTIMO_MES"] - ctl["DEV_REF_ANUAL"]
    ok_suma = ctl["DIF_SUMA"].abs().le(tol).all()
    ok_ult = ctl["DIF_ULTIMO"].abs().le(tol).all()
    if ok_suma:
        semantica, col = "MENSUAL (la suma de los meses cuadra con el anual)", "DIF_SUMA"
    elif ok_ult:
        semantica, col = "ACUMULADO (el último mes cuadra con el anual: NO sumar meses)", "DIF_ULTIMO"
    else:
        semantica, col = "NO CUADRA (revisar)", "DIF_SUMA"
    ctl["ESTADO"] = np.where(ctl[col].abs().le(tol), "OK", "REVISAR")
    if not completo:
        semantica = ("AÑO INCOMPLETO: la referencia anual incluye meses no raspados "
                     "(DIF = gasto de esos meses)")
        ctl["ESTADO"] = np.where(ctl[col].abs().le(tol), "OK", "PARCIAL")
    ctl["TOLERANCIA_S/"] = tol
    ctl["FECHA_EXTRACCION"] = date.today().isoformat()
    return ctl, semantica

# ── 5. CONEXIONES ─────────────────────────────────────────────────────────────
drive = conectar_drive([DRIVE_FOLDER_ID, DRIVE_FOLDER_CONTROL_ID]) if SUBIR_A_DRIVE else None

# Elige el formato de URL que devuelve la tabla de detalle (1 a 4 consultas)
FORMATO = None
if calibrar(AÑO, meses[0], departamentos[0]["codigo"], verbose=False) is None:
    raise RuntimeError("Consulta Amigable no devolvió la tabla de detalle con ningún formato de URL.")
print(f"🔗 Consulta Amigable OK (formato: {FORMATO['nombre']})\n")

# ── 6. BUCLE MES A MES (regiones en secuencia: Loreto -> San Martín -> Ucayali) ─
registros_todos, errores_totales, avisos_totales = [], [], []
archivos_generados, resumen_meses = [], []

print(f"Iniciando extracción {AÑO}: {len(meses)} mes(es) x {len(departamentos)} región(es).\n")

for mes in tqdm(meses, desc="Meses"):
    etiqueta = meses_map[mes]
    regs_mes, errs_mes = [], []
    for d in departamentos:
        regs, err, aviso = extraer(AÑO, mes, d["codigo"])
        regs_mes.extend(regs)
        if err:
            errs_mes.append({"AÑO": AÑO, "MES": etiqueta, "DEPARTAMENTO": d["departamento"], "ERROR": err})
        if aviso:
            avisos_totales.append({"AÑO": AÑO, "MES": etiqueta, "DEPARTAMENTO": d["departamento"], "AVISO": aviso})
        time.sleep(random.uniform(*PAUSA))

    errores_totales.extend(errs_mes)
    registros_todos.extend(regs_mes)
    df_mes = pd.DataFrame(regs_mes)
    if df_mes.empty:
        estado = "SIN DATOS"
    else:
        archivo = os.path.join(ruta_guardado, nombre_archivo_salida(AÑO, [mes]))
        df_mes[COLUMNAS_SALIDA].to_csv(archivo, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo)
        if SUBIR_A_DRIVE:
            subir_a_drive(drive, archivo, DRIVE_FOLDER_ID)
        estado = "OK" if not errs_mes else "INCOMPLETO (con errores)"
    resumen_meses.append({"MES": etiqueta, "FILAS": len(df_mes), "ERRORES": len(errs_mes), "ESTADO": estado})

print("═" * 70)
print(f"RESUMEN {AÑO}")
display(pd.DataFrame(resumen_meses))
if errores_totales:
    print("⚠️ Consultas con error:")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(errores_totales))
if avisos_totales:
    print("⚠️ Avisos de cuadre interno (suma del detalle ≠ fila del filtro):")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(avisos_totales))
if not errores_totales and not avisos_totales:
    print("✅ Sin errores y con cuadre interno en todas las consultas.")

# ── 7. CONTROL ANUAL + CIERRE (guardado, Drive, descarga) ────────────────────
ctl_anual = pd.DataFrame()
df_total = pd.DataFrame(registros_todos)
cubre_anio = (MES_INICIO == 1) and (AÑO < hoy.year or INCLUIR_MES_EN_CURSO)

if CONTROL_ANUAL and df_total.empty:
    print("\nControl anual omitido: no hay datos raspados.")
elif CONTROL_ANUAL:
    print("\n🔍 Control anual contra Consulta Amigable…")
    ctl_anual, semantica = control_anual(df_total, AÑO, meses[-1], completo=cubre_anio)
    print(f"   Semántica detectada del Devengado por mes: {semantica}")
    if not ctl_anual.empty:
        display(ctl_anual["ESTADO"].value_counts().to_frame("funciones"))
        display(ctl_anual[ctl_anual["ESTADO"] == "REVISAR"])
        display(ctl_anual.groupby("DEPARTAMENTO")[["DEV_REF_ANUAL", "DEV_SUMA_MESES", "DEV_ULTIMO_MES"]].sum())
        sufijo_ctl = "" if cubre_anio else f" - {etiqueta_meses(meses)}"
        archivo_ctl = os.path.join(ruta_guardado, f"CONTROL {NIVEL_CORTO} {AÑO}{sufijo_ctl}.csv")
        ctl_anual.to_csv(archivo_ctl, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo_ctl)
        if SUBIR_A_DRIVE:
            subir_a_drive(drive, archivo_ctl, DRIVE_FOLDER_CONTROL_ID)
    else:
        print("   ⚠️ No se pudo armar el control (sin referencia anual).")

if SUBIR_A_DRIVE and (errores_totales or avisos_totales):
    log = pd.concat([pd.DataFrame(errores_totales), pd.DataFrame(avisos_totales)], ignore_index=True)
    archivo_log = os.path.join(ruta_guardado, f"LOG {NIVEL_CORTO} {AÑO} - {etiqueta_meses(meses)}.csv")
    log.to_csv(archivo_log, index=False, encoding="utf-8-sig")
    subir_a_drive(drive, archivo_log, DRIVE_FOLDER_ID)

if SUBIR_A_DRIVE:
    print(f"\n☁️ DRIVE: {len(drive_subidos)} subido(s), {len(drive_fallidos)} fallido(s)")
    print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
    print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")

if not archivos_generados:
    print("\n⚠️ No hay archivos para descargar.")
elif DESCARGA_FINAL == "zip":
    from google.colab import files
    ruta_zip = os.path.join(ruta_guardado, nombre_archivo_salida(AÑO, meses).replace(".csv", ".zip"))
    with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for a in archivos_generados:
            z.write(a, arcname=os.path.basename(a))
    files.download(ruta_zip)
elif DESCARGA_FINAL == "individual":
    from google.colab import files
    for a in archivos_generados:
        files.download(a)
else:
    print(f"\n💾 Archivos guardados en {ruta_guardado}")